# Module 3 — Closed-Loop Control (runnable)

The error signal, a PID controller, and a simple step-response simulation.

## Setup
Run this first. It defines the machine and the formulas using only Python's standard library, so the notebook runs anywhere with no `pip install`.

In [ ]:
# PKM curriculum — runnable Python reference (standard library only, no installs needed)
from math import pi, sqrt, hypot

# machine defaults (identical to the simulation engine)
B, L_CLOSED, STROKE = 0.6, 0.4, 0.6
BORE, ROD = 0.040, 0.022
SUPPLY, RELIEF = 16e6, 21e6
PUMP_MAX_FLOW, RATED_FLOW, RATED_DP = 6e-4, 2.5e-4, 3.5e6
PAYLOAD = 12.0

def inverse_kinematics(x, y, b=B): return hypot(x+b, y), hypot(x-b, y)
def forward_kinematics(L1, L2, b=B):
    x = (L1**2 - L2**2)/(4*b); return x, sqrt(max(0.0, L1**2 - (x+b)**2))
def det_jacobian(x, y, b=B):
    L1, L2 = inverse_kinematics(x, y, b); return 2*b*y/(L1*L2)
def manipulability(x, y, b=B): return abs(det_jacobian(x, y, b))
def cap_area(D=BORE): return pi*D**2/4
def rod_area(D=BORE, d=ROD): return pi*(D**2 - d**2)/4
def asymmetry(D=BORE, d=ROD): return cap_area(D)/rod_area(D, d)
def valve_flow(u, dP, Qr=RATED_FLOW, dPr=RATED_DP): return u*Qr*sqrt(max(0.0, dP/dPr))
print("reference loaded — stdlib only")

### Lesson 1.1 — The error signal
e = r - y drives every correction.

In [ ]:
r, y = 0.90, 0.87
print(f'error e = {r - y:+.3f} m  (positive -> too short -> extend)')

### Lesson 1.2 — A PID controller
Derivative-on-measurement with an anti-windup clamp.

In [ ]:
class PID:
    def __init__(self, Kp, Ki, Kd, i_max=1.0):
        self.Kp, self.Ki, self.Kd, self.i_max = Kp, Ki, Kd, i_max
        self.integ = 0.0; self.prev = 0.0
    def update(self, error, meas, dt):
        self.integ = max(-self.i_max, min(self.i_max, self.integ + error*dt))
        deriv = -(meas - self.prev)/dt; self.prev = meas
        return self.Kp*error + self.Ki*self.integ + self.Kd*deriv
print("PID defined")

### Lesson 1.2/1.3 — Simulate a step response
Drive a first-order plant to a setpoint and watch it settle.

In [ ]:
def step_response(Kp, Ki, Kd, setpoint=1.0, dt=0.01, n=800, c=3.0):
    """Damped 2nd-order plant: high Kp overshoots, Kd damps it."""
    integ = prev = y = v = 0.0
    traj = []
    for _ in range(n):
        e = setpoint - y
        integ = max(-2, min(2, integ + e*dt))      # anti-windup clamp
        deriv = -(y - prev)/dt                       # derivative on measurement
        prev = y
        u = Kp*e + Ki*integ + Kd*deriv
        v += (u - c*v) * dt                          # force minus damping
        y += v * dt                                  # integrate to position
        traj.append(y)
    peak = max(traj)
    overshoot = max(0.0, (peak - setpoint)/setpoint)
    band = [i for i, val in enumerate(traj) if abs(val - setpoint) > 0.02]
    settle = (band[-1] + 1)*dt if band else 0.0
    return overshoot, settle

presets = {"too slow (low Kp)": (3,0,0),
           "too hot (high Kp, no Kd)": (60,0,0),
           "well tuned (Kp+Kd)": (30,0,8)}
for name,(Kp,Ki,Kd) in presets.items():
    os, ts = step_response(Kp,Ki,Kd)
    print(f"{name:26s}: overshoot={os*100:5.1f}%   settling={ts:.2f} s")

### Lesson 2.2 — Feedforward removes tracking lag
Pre-command the known velocity; feedback only trims the remainder.

In [ ]:
v_target = 0.10            # m/s, a ramping setpoint
Q_ff = v_target * cap_area()
print(f'feedforward flow for {v_target} m/s = {Q_ff*60000:.1f} L/min (sent before any error appears)')